# CNN Spatio-Temporal Transformer — Plan C
## Leave-One-Subject-Out (LOSO) Cross-Validation

**Architecture:** CNN (spatial-preserving) → Spatio-Temporal Positional Encoding → Transformer → mean pool → FC

---

## Plan C — 去掉全局池化，保留空间信息

### 问题背景（v1/v2/v3/双头共同的根本缺陷）

所有之前的版本都有一个相同的设计决策：

```
CNN → AdaptiveAvgPool2d(1) → (B*T, 32)  ← 空间信息在此全部丢失
```

`AdaptiveAvgPool2d(1)` 把整张鞋垫的 66×30 压力图压缩成一个向量，  
这意味着「前掌压力」和「后跟压力」的空间位置信息被完全抛弃。  
但是 **heel 的 3D 位置高度依赖后跟区域的压力分布**，而不是全鞋垫的平均压力。

### Plan C 的改动：每个空间格子成为一个独立 Token

```
[之前所有版本]
输入: (B, T, 4, 66, 30)
  → CNN → 全局池化 → (B, T, 32)        ← T 个 token，每个是全鞋垫的平均特征
  → Transformer 在 T=60 个 token 上做 self-attention

[Plan C]
输入: (B, T, 4, 66, 30)
  → CNN → 空间池化到 6×3 → (B, T, 32, 6, 3)  ← 保留 18 个空间格子
  → reshape → (B, T×18, 32)                    ← 1080 个 token
  → Transformer 在 T×Hp×Wp = 1080 个 token 上做 self-attention
```

### Token 数量对比

| 版本 | Token 数 | 每个 Token 代表 |
|------|---------|----------------|
| v1/v2/v3/双头 | T=60 | 一帧的全鞋垫平均特征 |
| **Plan C** | **T×6×3=1080** | **一帧的一个空间格子（约 11×10 像素区域）** |

Self-attention 现在可以学习「第 t 帧的后跟区格子」和「第 t+5 帧的前掌格子」之间的关系，  
而不是只看全鞋垫的时间序列。

### 位置编码设计

每个 token 有两个位置信息，分别用不同方式编码：
- **时间位置**：固定正弦编码（时间有自然连续顺序）
- **空间位置**：可学习 Embedding（鞋垫格子没有自然数值顺序）

### 代价

1080 个 token vs 60 个 token → Attention 计算量约 `(1080/60)² = 324×`。  
Batch size 建议从 32 降至 **16** 以控制显存。

> **Note:** Subjects 1 和 3 因电池时间戳漂移被排除。LOSO 在剩余 3 个 subjects 上运行。

## 1. Imports and paths

In [ ]:
from pathlib import Path
import random
import math
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

BASE       = Path('data')
SYNCED_DIR = BASE / 'synced'
MODELS_DIR = BASE / 'models_transformer_planC'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Device:', DEVICE)
print('Synced dir:', SYNCED_DIR)
print('Models dir:', MODELS_DIR)

## 2. Joint setup and skeleton connections

In [ ]:
TARGET_JOINTS = [
    'left-hip',
    'right-hip',
    'left-knee',
    'right-knee',
    'left-heel',
    'right-heel',
    'left-big-toe-tip',
    'right-big-toe-tip',
]
OUTPUT_JOINTS = TARGET_JOINTS.copy()

CONNECTIONS = [
    ('left-hip',   'left-knee'),
    ('left-knee',  'left-heel'),
    ('left-heel',  'left-big-toe-tip'),
    ('right-hip',  'right-knee'),
    ('right-knee', 'right-heel'),
    ('right-heel', 'right-big-toe-tip'),
    ('left-hip',   'right-hip'),
]

def as_str_list(names):
    return [n.decode('utf-8') if isinstance(n, bytes) else str(n) for n in names]

print(f'Output joints ({len(OUTPUT_JOINTS)}): {OUTPUT_JOINTS}')

## 3. Load synced sessions

In [ ]:
def load_synced_sessions(synced_dir):
    session_files = sorted(synced_dir.glob('*_synced.npz'))
    if not session_files:
        raise FileNotFoundError(f'No synced files found in {synced_dir}')
    sessions = []
    for path in session_files:
        data = np.load(path, allow_pickle=True)
        insoles     = np.asarray(data['insoles'],     dtype=np.float32)
        joints      = np.asarray(data['joints'],      dtype=np.float32)
        joint_names = as_str_list(data['joint_names'])
        timestamps  = np.asarray(data['timestamps'],  dtype=np.float64)
        sessions.append({
            'name':        path.stem.replace('_synced', ''),
            'insoles':     insoles,
            'joints':      joints,
            'joint_names': joint_names,
            'timestamps':  timestamps,
        })
        print(f'{path.name}: insoles={insoles.shape}, joints={joints.shape}')
    return sessions

sessions = load_synced_sessions(SYNCED_DIR)
print(f'\nTotal subjects loaded: {len(sessions)}')
assert len(sessions) >= 2, f'Need at least 2 subjects for LOSO, found {len(sessions)}'

## 4. Session preprocessing

In [ ]:
def replace_nan_with_zero(x):
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def build_joint_index(joint_names):
    mapping = {name: i for i, name in enumerate(joint_names)}
    missing = [name for name in TARGET_JOINTS if name not in mapping]
    if missing:
        raise ValueError(f'Missing required joints: {missing}')
    return mapping

def select_output_targets(joints, joint_names):
    name_to_idx  = build_joint_index(joint_names)
    keep_indices = [name_to_idx[name] for name in OUTPUT_JOINTS]
    return joints[:, keep_indices, :].astype(np.float32)

def preprocess_sessions(sessions):
    processed = []
    for session in sessions:
        x = replace_nan_with_zero(session['insoles'])
        y = select_output_targets(session['joints'], session['joint_names'])
        processed.append({
            'name':       session['name'],
            'X':          x,
            'Y':          y,
            'Y_raw':      y.copy(),
            'timestamps': session['timestamps'],
        })
        print(f"{session['name']}: X={x.shape}, Y={y.shape}")
    return processed

processed_sessions = preprocess_sessions(sessions)

## 5. Model configuration

Plan C 新增两个超参数：`POOLED_SIZE` 和 `MAX_T`。

In [ ]:
SEQ_LEN     = 60
SEQ_STRIDE  = 1
BATCH_SIZE  = 16    # reduced from 32: Plan C has 1080 tokens vs 60, higher memory
EPOCHS      = 400
PATIENCE    = 20
VAL_RATIO   = 0.15

# CNN encoder
CNN_CHANNELS = (16, 32)

# [Plan C] Spatial pooling target — CNN output is pooled to Hp×Wp grid
# before being fed to the Transformer as spatial tokens.
# 6×3 = 18 spatial cells per frame → each covers ~11×10 insole pixels.
# Larger values = more spatial detail but quadratically more tokens.
POOLED_SIZE = (6, 3)   # (Hp, Wp)
MAX_T       = 64       # positional encoding buffer size, must be >= SEQ_LEN

# Derived: number of tokens per sequence
N_SPATIAL   = POOLED_SIZE[0] * POOLED_SIZE[1]          # 18
N_TOKENS    = SEQ_LEN * N_SPATIAL                       # 1080

# Transformer
D_MODEL                = 128
NHEAD                  = 4
NUM_TRANSFORMER_LAYERS = 2
DIM_FEEDFORWARD        = 256
DROPOUT                = 0.2

print('Config OK')
print(f'  SEQ_LEN={SEQ_LEN}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, PATIENCE={PATIENCE}')
print(f'  POOLED_SIZE={POOLED_SIZE} → N_SPATIAL={N_SPATIAL} spatial cells per frame')
print(f'  N_TOKENS per sequence = {SEQ_LEN} × {N_SPATIAL} = {N_TOKENS}')
print(f'  (vs {SEQ_LEN} tokens in v3: Plan C is {N_SPATIAL}× more tokens)')
print(f'  D_MODEL={D_MODEL}, NHEAD={NHEAD}, LAYERS={NUM_TRANSFORMER_LAYERS}, FFN={DIM_FEEDFORWARD}')

## 6. Build temporal sequences

与 v3 完全相同：速度差分通道，输入为 (T, 4, H, W)。

In [ ]:
class SlidingWindowDataset(Dataset):
    def __init__(self, sessions, items, seq_len):
        self.sessions = sessions
        self.items    = items
        self.seq_len  = seq_len

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        session_idx, end_idx = self.items[idx]
        s         = self.sessions[session_idx]
        start_idx = end_idx - self.seq_len + 1

        x = s['X'][start_idx : end_idx + 1]          # (T, 2, H, W)
        x_diff = np.diff(x, axis=0, prepend=x[:1])   # (T, 2, H, W) velocity diff
        x_input = np.concatenate([x, x_diff], axis=1).astype(np.float32)  # (T, 4, H, W)

        y     = s['Y'][end_idx]
        y_raw = s['Y_raw'][end_idx]
        return {
            'X':       torch.from_numpy(x_input).float(),
            'Y':       torch.from_numpy(y).float(),
            'Y_raw':   torch.from_numpy(y_raw).float(),
            'session': s['name'],
        }


def build_items(sessions_subset, seq_len):
    items = []
    for local_i, s in enumerate(sessions_subset):
        T = len(s['X'])
        if T < seq_len:
            print(f"  WARNING: {s['name']} has only {T} frames < SEQ_LEN={seq_len}, skipped.")
            continue
        for end_idx in range(seq_len - 1, T):
            items.append((local_i, end_idx))
    return items

print('Dataset helpers defined.')
print('Input: (T, 4, H, W) — 4 channels: 2 pressure + 2 velocity diff')

## 7. CNN Spatio-Temporal Transformer — Plan C

### 完整架构图

```
输入 (B, T=60, 4, 66, 30)
  │
  └─ 对每一帧独立运行 FrameEncoderSpatial
       ├─ Conv2d(4→16) → BN → ReLU
       ├─ Conv2d(16→16) → BN → ReLU
       ├─ MaxPool2d(2)   →  (B*T, 16, 33, 15)
       ├─ Conv2d(16→32) → BN → ReLU
       ├─ Conv2d(32→32) → BN → ReLU
       └─ AdaptiveAvgPool2d(6, 3)  →  (B*T, 32, 6, 3)   ← 保留 6×3=18 空间格子
  │
  reshape → (B, T, 32, 6, 3)
  permute → (B, T, 6, 3, 32)
  reshape → (B, T×6×3, 32) = (B, 1080, 32)               ← 1080 个 spatio-temporal token
  │
  └─ Linear(32 → D_MODEL=128)  →  (B, 1080, 128)
  │
  └─ Spatio-Temporal Positional Encoding
       ├─ 时间位置: 固定正弦编码（pe_t[t]，t=0..59）
       └─ 空间位置: 可学习 Embedding（spatial_embed[s]，s=0..17）
       每个 token 得到: pe_t[t] + spatial_embed[s]
  │
  └─ TransformerEncoder (2 layers, nhead=4)
       Self-attention 跨越 1080 个 token（时间 × 空间）
  │
  └─ LayerNorm → Mean pool over 1080 tokens → (B, 128)
  │
  └─ Dropout → Linear(128 → 24) → 8 joints × xyz
```

### 关键创新点

**之前（v1-v3 + 双头）：** Transformer 的每个 token = 一帧的全鞋垫平均，  
空间位置（前掌 vs 后跟）的信息在进入 Transformer 之前就被彻底丢弃。

**Plan C：** Transformer 的每个 token = 一帧的一个空间格子（6×3 网格之一），  
Self-attention 可以学习「第 t 帧后跟区格子的压力」和「第 t+k 帧前掌区格子的压力」之间的关系，  
这正是 heel/toe 预测所需要的空间-时间联合信息。

In [ ]:
# ── CNN encoder that PRESERVES spatial layout ───────────────
class FrameEncoderSpatial(nn.Module):
    '''
    Same conv stack as v3 FrameEncoder, but NO AdaptiveAvgPool2d(1).
    Instead, pool to a small spatial grid (Hp, Wp) and keep the 2D layout.

    Input : (B, in_channels, H, W)     e.g. (B, 4, 66, 30)
    Output: (B, c2, Hp, Wp)            e.g. (B, 32, 6, 3)
    '''
    def __init__(self, in_channels=4, c1=16, c2=32, pooled_size=(6, 3)):
        super().__init__()
        self.pooled_size = pooled_size
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, c1, kernel_size=3, padding=1),
            nn.BatchNorm2d(c1), nn.ReLU(),
            nn.Conv2d(c1, c1, kernel_size=3, padding=1),
            nn.BatchNorm2d(c1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(c1, c2, kernel_size=3, padding=1),
            nn.BatchNorm2d(c2), nn.ReLU(),
            nn.Conv2d(c2, c2, kernel_size=3, padding=1),
            nn.BatchNorm2d(c2), nn.ReLU(),
            # No AdaptiveAvgPool2d(1) — that is the key change
        )
        self.spatial_pool = nn.AdaptiveAvgPool2d(pooled_size)

    def forward(self, x):
        x = self.encoder(x)          # (B, c2, ~33, ~15)
        x = self.spatial_pool(x)     # (B, c2, Hp, Wp)  — spatial layout preserved
        return x


# ── Spatio-temporal positional encoding ─────────────────────
class SpatioTemporalPositionalEncoding(nn.Module):
    '''
    Adds position information along BOTH the time axis and the spatial axis.

    Each token represents a (timestep t, spatial cell s) pair:
      pos(t, s) = temporal_pe[t]  +  spatial_embed[s]

    Temporal: fixed sinusoidal  — time has a natural continuous ordering.
    Spatial:  learned embedding — insole grid cells have no inherent
              numerical ordering, so learning it from data is better.
    '''
    def __init__(self, d_model, max_T=64, Hp=6, Wp=3, dropout=0.1):
        super().__init__()
        self.dropout   = nn.Dropout(dropout)
        self.Hp, self.Wp = Hp, Wp
        self.n_spatial = Hp * Wp

        # Fixed sinusoidal temporal PE (Vaswani et al., 2017)
        pe_t = torch.zeros(max_T, d_model)
        pos  = torch.arange(0, max_T).unsqueeze(1).float()
        div  = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe_t[:, 0::2] = torch.sin(pos * div)
        pe_t[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe_t', pe_t)   # (max_T, d_model)

        # Learned spatial embedding — one vector per grid cell
        self.spatial_embed = nn.Embedding(self.n_spatial, d_model)

    def forward(self, x, T):
        # x: (B, T * n_spatial, d_model)
        # tokens ordered: (t=0, s=0), (t=0, s=1), ..., (t=0, s=N-1), (t=1, s=0), ...
        device = x.device
        t_idx = torch.arange(T, device=device).repeat_interleave(self.n_spatial)  # (L,)
        s_idx = torch.arange(self.n_spatial, device=device).repeat(T)             # (L,)

        pos = self.pe_t[t_idx] + self.spatial_embed(s_idx)   # (L, d_model)
        x = x + pos.unsqueeze(0)                              # broadcast over batch
        return self.dropout(x)


# ── Full spatio-temporal model (Plan C) ─────────────────────
class CNNTransformerSpatioTemporal(nn.Module):
    '''
    Plan C: Transformer operates over T * Hp * Wp spatio-temporal tokens.

    Each token = (one timestep, one spatial grid cell).
    Self-attention can now learn cross-time AND cross-space dependencies,
    e.g. the heel region at frame t correlating with toe region at frame t+5.
    '''
    def __init__(
        self,
        output_dim,
        in_channels=4,
        c1=16,
        c2=32,
        pooled_size=(6, 3),
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.2,
        max_T=64,
    ):
        super().__init__()
        self.Hp, self.Wp = pooled_size
        self.n_spatial   = self.Hp * self.Wp

        self.cnn = FrameEncoderSpatial(in_channels, c1, c2, pooled_size)

        # c2 is the output channel of CNN, project to d_model
        self.input_proj = nn.Linear(c2, d_model)

        self.pos_enc = SpatioTemporalPositionalEncoding(
            d_model, max_T=max_T, Hp=self.Hp, Wp=self.Wp, dropout=dropout
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm    = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(d_model, output_dim)

    def forward(self, x):
        B, T, C, H, W = x.shape

        # Per-frame CNN (spatial-preserving)
        x = x.view(B * T, C, H, W)
        x = self.cnn(x)                               # (B*T, c2, Hp, Wp)

        # Flatten to spatio-temporal token sequence
        x = x.view(B, T, -1, self.Hp, self.Wp)        # (B, T, c2, Hp, Wp)
        x = x.permute(0, 1, 3, 4, 2)                  # (B, T, Hp, Wp, c2)
        x = x.reshape(B, T * self.Hp * self.Wp, -1)   # (B, T*Hp*Wp, c2)

        # Project + add spatio-temporal positions
        x = self.input_proj(x)                        # (B, L, d_model)
        x = self.pos_enc(x, T=T)                      # position: time + space

        # Transformer attends over all L = T*Hp*Wp tokens
        x = self.transformer(x)                       # (B, L, d_model)
        x = self.norm(x)

        # Pool over all tokens (time and space together)
        x = self.dropout(x.mean(dim=1))               # (B, d_model)
        return self.fc(x)                             # (B, output_dim)


# ── Factory helpers ──────────────────────────────────────────
def make_model():
    output_dim = len(OUTPUT_JOINTS) * 3
    m = CNNTransformerSpatioTemporal(
        output_dim      = output_dim,
        in_channels     = 4,
        c1              = CNN_CHANNELS[0],
        c2              = CNN_CHANNELS[1],
        pooled_size     = POOLED_SIZE,
        d_model         = D_MODEL,
        nhead           = NHEAD,
        num_layers      = NUM_TRANSFORMER_LAYERS,
        dim_feedforward = DIM_FEEDFORWARD,
        dropout         = DROPOUT,
        max_T           = MAX_T,
    ).to(DEVICE)
    return m


def make_optimizer(model):
    return torch.optim.Adam([
        {'params': model.cnn.parameters(),         'lr': 2e-3},
        {'params': model.input_proj.parameters(),  'lr': 1e-3},
        {'params': model.pos_enc.parameters(),     'lr': 1e-3},
        {'params': model.transformer.parameters(), 'lr': 5e-4},
        {'params': model.fc.parameters(),          'lr': 1e-3},
    ], weight_decay=1e-4)


# Sanity check
_m = make_model()
_n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
_n_tokens  = SEQ_LEN * _m.n_spatial
print(f'Trainable parameters : {_n_params:,}')
print(f'Tokens per sequence  : {_n_tokens}  (T={SEQ_LEN} × spatial={_m.n_spatial})')
_dummy = torch.zeros(2, SEQ_LEN, 4, 66, 30, device=DEVICE)
_out   = _m(_dummy)
print(f'Output shape         : {tuple(_out.shape)}  (expected: (2, {len(OUTPUT_JOINTS) * 3}))')
del _m, _dummy, _out

## 8. Loss function and training helpers

In [ ]:
# Per-joint weights — same as v2/v3, give more weight to distal joints
JOINT_WEIGHTS = torch.tensor([
    0.5, 0.5,   # hips  — easiest to predict
    1.0, 1.0,   # knees
    2.0, 2.0,   # heels — hardest (swing phase: zero insole signal)
    2.0, 2.0,   # toes
], dtype=torch.float32)


def weighted_smoothl1(pred, target):
    pred_j    = pred.view(-1, len(OUTPUT_JOINTS), 3)
    target_j  = target.view(-1, len(OUTPUT_JOINTS), 3)
    per_joint = torch.abs(pred_j - target_j).mean(dim=2)   # (B, 8)
    w = JOINT_WEIGHTS.to(pred.device)
    return (per_joint * w).mean()


criterion = weighted_smoothl1


def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, n_samples = 0.0, 0
    with torch.set_grad_enabled(is_train):
        for batch in loader:
            x    = batch['X'].to(DEVICE)
            y    = batch['Y'].to(DEVICE).view(x.size(0), -1)
            pred = model(x)
            loss = criterion(pred, y)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            bs = x.size(0)
            total_loss += loss.item() * bs
            n_samples  += bs
    return total_loss / max(n_samples, 1)


def evaluate(model, loader, y_mean=None, y_std=None):
    model.eval()
    total_loss, n_samples = 0.0, 0
    preds_list, trues_list = [], []
    with torch.no_grad():
        for batch in loader:
            x     = batch['X'].to(DEVICE)
            y     = batch['Y'].to(DEVICE).view(x.size(0), -1)
            y_raw = batch['Y_raw'].cpu().numpy()
            pred  = model(x)
            loss  = criterion(pred, y)
            bs    = x.size(0)
            total_loss += loss.item() * bs
            n_samples  += bs
            pred_raw = pred.view(bs, len(OUTPUT_JOINTS), 3).cpu().numpy()
            if y_mean is not None and y_std is not None:
                pred_raw = pred_raw * y_std + y_mean
            preds_list.append(pred_raw)
            trues_list.append(y_raw)

    preds = np.concatenate(preds_list, axis=0)
    trues = np.concatenate(trues_list, axis=0)
    mse   = float(np.mean((preds - trues) ** 2))
    mae   = float(np.mean(np.abs(preds - trues)))
    per_joint_mae = np.mean(np.abs(preds - trues), axis=(0, 2))
    return {
        'loss':          total_loss / max(n_samples, 1),
        'mse':           mse,
        'mae':           mae,
        'per_joint_mae': per_joint_mae,
        'preds':         preds,
        'trues':         trues,
    }

print('Helpers defined.')

## 9. LOSO Cross-Validation

**Protocol:** Same as all previous versions — fold k uses subject k as test, rest as train.

In [ ]:
n_subjects     = len(processed_sessions)
fold_results   = []
fold_histories = []

print('=' * 60)
print('CNN Spatio-Temporal Transformer (Plan C) — LOSO')
print(f'Key change: token count {SEQ_LEN} → {N_TOKENS} (T × {N_SPATIAL} spatial cells)')
print(f'POOLED_SIZE={POOLED_SIZE}, BATCH_SIZE={BATCH_SIZE}')
print('=' * 60)

for test_subj_idx in range(n_subjects):

    test_name   = processed_sessions[test_subj_idx]['name']
    train_idxs  = [i for i in range(n_subjects) if i != test_subj_idx]
    train_names = [processed_sessions[i]['name'] for i in train_idxs]

    print(f"\n{'─' * 60}")
    print(f'Fold {test_subj_idx + 1}/{n_subjects}  |  '
          f'Test: [{test_name}]  |  Train: {train_names}')
    print(f"{'─' * 60}")

    # ── Session subsets ──────────────────────────────────────
    train_sessions = [copy.deepcopy(processed_sessions[i]) for i in train_idxs]
    test_sessions  = [copy.deepcopy(processed_sessions[test_subj_idx])]

    train_Y = np.concatenate([s['Y'] for s in train_sessions], axis=0)
    Y_mean  = train_Y.mean(axis=0)
    Y_std   = train_Y.std(axis=0) + 1e-6
    for s in train_sessions:
        s['Y'] = (s['Y'] - Y_mean) / Y_std
    for s in test_sessions:
        s['Y'] = (s['Y'] - Y_mean) / Y_std

    # ── Sliding-window items ─────────────────────────────────
    all_train_items = build_items(train_sessions, SEQ_LEN)
    test_items      = build_items(test_sessions,  SEQ_LEN)

    random.seed(SEED)
    random.shuffle(all_train_items)
    n_val            = max(1, int(len(all_train_items) * VAL_RATIO))
    val_items_fold   = all_train_items[-n_val:]
    train_items_fold = all_train_items[:-n_val]

    print(f'  Train sequences : {len(train_items_fold)}')
    print(f'  Val   sequences : {len(val_items_fold)}')
    print(f'  Test  sequences : {len(test_items)}')

    # ── DataLoaders ──────────────────────────────────────────
    kw = dict(batch_size=BATCH_SIZE, num_workers=0,
              pin_memory=torch.cuda.is_available())
    train_loader = DataLoader(
        SlidingWindowDataset(train_sessions, train_items_fold, SEQ_LEN),
        shuffle=True, **kw)
    val_loader = DataLoader(
        SlidingWindowDataset(train_sessions, val_items_fold, SEQ_LEN),
        shuffle=False, **kw)
    test_loader = DataLoader(
        SlidingWindowDataset(test_sessions, test_items, SEQ_LEN),
        shuffle=False, **kw)

    # ── Fresh model ──────────────────────────────────────────
    torch.manual_seed(SEED)
    model     = make_model()
    optimizer = make_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-5
    )
    model_path = MODELS_DIR / f'cnn_transformer_planC_fold{test_subj_idx+1}_{test_name}.pt'

    # ── Training loop with early stopping ───────────────────
    history       = {'train_loss': [], 'val_loss': [], 'lr': []}
    best_val      = float('inf')
    best_epoch    = -1
    patience_left = PATIENCE

    for epoch in range(1, EPOCHS + 1):
        train_loss = run_epoch(model, train_loader, optimizer=optimizer)
        val_loss   = run_epoch(model, val_loader,   optimizer=None)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['lr'].append(current_lr)

        show_lr = (epoch == 1 or epoch % 50 == 0)
        print(f'  Epoch {epoch:03d}/{EPOCHS} | '
              f'train={train_loss:.6f} | val={val_loss:.6f}'
              + (f' | lr={current_lr:.2e}' if show_lr else ''), end='')

        if val_loss < best_val:
            best_val      = val_loss
            best_epoch    = epoch
            patience_left = PATIENCE
            torch.save(model.state_dict(), model_path)
            print('  ✓', end='')
        else:
            patience_left -= 1
        print()

        scheduler.step()

        if patience_left <= 0:
            print(f'  Early stopping at epoch {epoch}.')
            break

    print(f'  Best epoch: {best_epoch},  best val loss: {best_val:.6f}')

    # ── Evaluate on test set ─────────────────────────────────
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    results = evaluate(model, test_loader, y_mean=Y_mean, y_std=Y_std)

    print(f'\n  ── Fold {test_subj_idx+1} Test Results ──')
    print(f'  Test subject : {test_name}')
    print(f'  Test MSE     : {results["mse"]:.6f}')
    print(f'  Test MAE     : {results["mae"]:.6f}')
    print(f'  Per-joint MAE:')
    for jname, jmae in zip(OUTPUT_JOINTS, results['per_joint_mae']):
        print(f'    {jname:<22s}: {jmae:.4f}')

    fold_results.append({
        'fold':       test_subj_idx + 1,
        'test_name':  test_name,
        'best_epoch': best_epoch,
        'best_val':   best_val,
        **{k: results[k] for k in ('loss', 'mse', 'mae', 'per_joint_mae', 'preds', 'trues')},
    })
    fold_histories.append(history)

## 10. Aggregated results

In [ ]:
mses    = [r['mse'] for r in fold_results]
maes    = [r['mae'] for r in fold_results]
pj_maes = np.stack([r['per_joint_mae'] for r in fold_results])

print('=' * 60)
print('CNN Spatio-Temporal Transformer (Plan C) — Aggregated Results')
print('=' * 60)

print(f"\n{'Fold':<6} {'Subject':<16} {'MSE':>10} {'MAE':>10}")
print('─' * 46)
for r in fold_results:
    print(f"  {r['fold']:<4} {r['test_name']:<16} "
          f"{r['mse']:>10.6f} {r['mae']:>10.6f}")
print('─' * 46)
print(f"  {'Mean':<20} {np.mean(mses):>10.6f} {np.mean(maes):>10.6f}")
print(f"  {'Std':<20} {np.std(mses):>10.6f}  {np.std(maes):>10.6f}")

print('\nPer-joint MAE — mean ± std across folds:')
for j, jname in enumerate(OUTPUT_JOINTS):
    vals = pj_maes[:, j]
    print(f'  {jname:<22s}: {vals.mean():.4f} ± {vals.std():.4f}')

## 11. Loss curves

In [ ]:
fig, axes = plt.subplots(1, len(fold_histories), figsize=(5 * len(fold_histories), 4))
if len(fold_histories) == 1:
    axes = [axes]

for ax, history, r in zip(axes, fold_histories, fold_results):
    ep = range(1, len(history['train_loss']) + 1)
    ax.plot(ep, history['train_loss'], label='train')
    ax.plot(ep, history['val_loss'],   label='val')
    ax.axvline(r['best_epoch'], color='red', linestyle='--', linewidth=0.8,
               label=f"best ep {r['best_epoch']}")
    ax.set_title(f"Fold {r['fold']} — test: {r['test_name']}")
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('CNN Spatio-Temporal Transformer (Plan C) — Training Curves', fontweight='bold')
plt.tight_layout()
plt.savefig('planC_loss_curves.png', dpi=150)
plt.show()

## 12. Per-joint MAE bar chart

In [ ]:
mean_pj = pj_maes.mean(axis=0)
std_pj  = pj_maes.std(axis=0)
short_names = [j.replace('left-', 'L-').replace('right-', 'R-')
                .replace('-big-toe-tip', 'Toe') for j in OUTPUT_JOINTS]

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(OUTPUT_JOINTS))
ax.bar(x, mean_pj, yerr=std_pj, capsize=4, color='steelblue', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=30, ha='right')
ax.set_ylabel('MAE (normalised coords)')
ax.set_title('Plan C — Per-joint MAE (mean ± std over folds)')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('planC_per_joint_mae.png', dpi=150)
plt.show()

## 13. Skeleton visualisation and trajectory plots

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

CONN_IDX = [
    (0, 2), (2, 4), (4, 6),
    (1, 3), (3, 5), (5, 7),
    (0, 1),
]


def plot_skeleton_comparison(preds, trues, frame_idx=0, fold_name='subject'):
    fig = plt.figure(figsize=(14, 5))
    titles     = ['Ground Truth', 'Prediction', 'Overlay (GT=blue, Pred=red)']
    data_pairs = [
        (trues[frame_idx], None),
        (preds[frame_idx], None),
        (trues[frame_idx], preds[frame_idx]),
    ]
    for col, (title, (d1, d2)) in enumerate(zip(titles, data_pairs)):
        ax = fig.add_subplot(1, 3, col + 1, projection='3d')

        def draw(pts, color, alpha=1.0):
            ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=color, s=50, alpha=alpha)
            for i, j in CONN_IDX:
                ax.plot([pts[i,0], pts[j,0]], [pts[i,1], pts[j,1]],
                        [pts[i,2], pts[j,2]], c=color, lw=2, alpha=alpha)

        draw(d1, '#2196F3')
        if d2 is not None:
            draw(d2, '#F44336', alpha=0.7)
        ax.set_title(f'{title}\n[{fold_name}] frame {frame_idx}', fontsize=9)
        ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
        ax.view_init(elev=15, azim=-70)
    plt.tight_layout()
    plt.savefig(f'planC_skeleton_{fold_name}.png', dpi=150)
    plt.show()


def plot_joint_trajectories(preds, trues, fold_name='subject', n_frames=300):
    preds, trues = preds[:n_frames], trues[:n_frames]
    t = np.arange(len(preds))
    fig, axes = plt.subplots(8, 3, figsize=(15, 20))
    fig.suptitle(f'Joint Trajectories — [{fold_name}]  (Plan C)', fontsize=12, fontweight='bold')
    for j, jname in enumerate(OUTPUT_JOINTS):
        short = jname.replace('left-', 'L-').replace('right-', 'R-')
        for ax_i, ax_n in enumerate(['X', 'Y', 'Z']):
            ax = axes[j, ax_i]
            ax.plot(t, trues[:, j, ax_i], 'b-',  lw=1.5, label='GT')
            ax.plot(t, preds[:, j, ax_i], 'r--', lw=1.2, label='Pred')
            if ax_i == 0:
                ax.set_ylabel(f'{short}\n{ax_n}', fontsize=7)
            if j == 0 and ax_i == 0:
                ax.legend(fontsize=6)
            ax.tick_params(labelsize=6)
            ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'planC_trajectories_{fold_name}.png', dpi=150)
    plt.show()


def plot_error_heatmap(preds, trues, fold_name='subject'):
    per_frame_mae = np.mean(np.abs(preds - trues), axis=2)
    short = [jn.replace('left-','L-').replace('right-','R-').replace('-big-toe-tip','Toe')
             for jn in OUTPUT_JOINTS]
    fig, ax = plt.subplots(figsize=(16, 4))
    im = ax.imshow(per_frame_mae.T, aspect='auto', cmap='RdYlGn_r')
    ax.set_yticks(range(8))
    ax.set_yticklabels(short, fontsize=9)
    ax.set_xlabel('Frame')
    ax.set_title(f'Per-frame Per-joint MAE — [{fold_name}]  (Plan C)')
    plt.colorbar(im, ax=ax, label='MAE')
    plt.tight_layout()
    plt.savefig(f'planC_heatmap_{fold_name}.png', dpi=150)
    plt.show()


for result in fold_results:
    name  = result['test_name']
    preds = result['preds']
    trues = result['trues']
    print(f'\n=== Visualising [{name}] ===')
    plot_skeleton_comparison(preds, trues, frame_idx=50, fold_name=name)
    plot_joint_trajectories(preds, trues, fold_name=name, n_frames=300)
    plot_error_heatmap(preds, trues, fold_name=name)

## 14. LaTeX summary table

In [ ]:
print(r'\begin{table}[h]')
print(r'\centering')
print(r'\begin{tabular}{lcc}')
print(r'\toprule')
print(r'Test Subject & MSE & MAE \\')
print(r'\midrule')
for r in fold_results:
    print(f"  {r['test_name']} (Fold {r['fold']}) & "
          f"{r['mse']:.4f} & {r['mae']:.4f} \\\\")
print(r'\midrule')
print(f'  Mean $\\pm$ Std & '
      f'{np.mean(mses):.4f} $\\pm$ {np.std(mses):.4f} & '
      f'{np.mean(maes):.4f} $\\pm$ {np.std(maes):.4f} \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\caption{CNN Spatio-Temporal Transformer (Plan C) LOSO results.}')
print(r'\label{tab:planC_loso}')
print(r'\end{table}')